# Descriptive Statistics for DCE and Covariates

This notebook creates two appendix-style descriptive outputs:

1. Descriptive statistics for selected DCE variables and nonzero-value density plots for alternative DCE functional forms.
2. Descriptive statistics for unit-level characteristics and raw, non-imputed country-year control variables.

The country-year controls are reported using the pre-imputation versions with the `_ni` suffix when available.

In [1]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    from scipy.stats import gaussian_kde
    SCIPY_AVAILABLE = True
except ImportError:
    SCIPY_AVAILABLE = False
    warnings.warn("scipy is not available; density plots will use matplotlib histograms instead.")

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 240)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

In [ ]:
# Locate project root
current = Path.cwd().resolve()
while current.name != "Data_center_and_fossil_energy_Replication" and current.parent != current:
    current = current.parent

if current.name != "Data_center_and_fossil_energy_Replication":
    # Fallback for running this notebook from another directory
    current = Path("/Users/lichenglin/Library/CloudStorage/OneDrive-HKUST(Guangzhou)/Data_center_and_fossil_energy_Replication")

BASE = current
RAW = BASE / "Data" / "raw"
TEMP = BASE / "Data" / "temp"
USE = BASE / "Data" / "use"
FIGURES = BASE / "Results" / "Figures"
TABLES = BASE / "Results" / "Tables"

FIGURES.mkdir(parents=True, exist_ok=True)
TABLES.mkdir(parents=True, exist_ok=True)

DATA_PATH = USE / "gem_coal_plants_multi_record_sa_dce_robust_forR_clean.dta"

print(f"Project root: {BASE}")
print(f"Data path: {DATA_PATH}")

In [ ]:
df = pd.read_stata(DATA_PATH, convert_categoricals=False)
print(f"Loaded analysis data: {df.shape[0]:,} rows x {df.shape[1]:,} columns")

## Helper Functions

In [7]:
def check_vars(data, vars_needed, label="variables"):
    missing = [v for v in vars_needed if v not in data.columns]
    if missing:
        raise KeyError(f"Missing {label}: {missing}")


def summarize_numeric(data, specs):
    """Create a compact Nature-style descriptive table for numeric variables."""
    rows = []
    total_n = len(data)

    for item in specs:
        panel = item["panel"]
        variable = item["variable"]
        source_variable = item.get("source_variable", variable)
        description = item["description"]

        x = pd.to_numeric(data[source_variable], errors="coerce")
        x_nonmiss = x.dropna()
        x_positive = x_nonmiss[x_nonmiss > 0]

        if x_nonmiss.empty:
            stats = {k: np.nan for k in [
                "Mean", "SD", "Min", "P25", "Median", "P75", "P90", "P95", "P99", "Max",
                "Positive_mean", "Positive_median", "Positive_p95", "Positive_p99"
            ]}
        else:
            stats = {
                "Mean": x_nonmiss.mean(),
                "SD": x_nonmiss.std(ddof=1),
                "Min": x_nonmiss.min(),
                "P25": x_nonmiss.quantile(0.25),
                "Median": x_nonmiss.quantile(0.50),
                "P75": x_nonmiss.quantile(0.75),
                "P90": x_nonmiss.quantile(0.90),
                "P95": x_nonmiss.quantile(0.95),
                "P99": x_nonmiss.quantile(0.99),
                "Max": x_nonmiss.max(),
            }

            if x_positive.empty:
                stats.update({
                    "Positive_mean": np.nan,
                    "Positive_median": np.nan,
                    "Positive_p95": np.nan,
                    "Positive_p99": np.nan,
                })
            else:
                stats.update({
                    "Positive_mean": x_positive.mean(),
                    "Positive_median": x_positive.quantile(0.50),
                    "Positive_p95": x_positive.quantile(0.95),
                    "Positive_p99": x_positive.quantile(0.99),
                })

        rows.append({
            "Panel": panel,
            "Variable": variable,
            "Source_variable": source_variable,
            "Description": description,
            "N_total": total_n,
            "N_nonmissing": int(x_nonmiss.shape[0]),
            "Missing_N": int(total_n - x_nonmiss.shape[0]),
            "Nonmissing_share": x_nonmiss.shape[0] / total_n if total_n else np.nan,
            "Nonzero_N": int(x_positive.shape[0]),
            "Nonzero_share": x_positive.shape[0] / x_nonmiss.shape[0] if x_nonmiss.shape[0] else np.nan,
            **stats,
        })

    return pd.DataFrame(rows)


def savefig(fig, name, width=7.0, height=4.2):
    pdf_path = FIGURES / f"{name}.pdf"
    tif_path = FIGURES / f"{name}.tif"
    fig.set_size_inches(width, height)
    fig.savefig(pdf_path, bbox_inches="tight")
    fig.savefig(tif_path, bbox_inches="tight", dpi=300)
    print(f"Saved: {pdf_path}")
    print(f"Saved: {tif_path}")

## 1. DCE Descriptive Statistics

The variables are ordered as follows: baseline 25 km DCE by construction period, alternative spatial definitions for the 2020-2024 exposure, administrative-boundary exposure, and larger-data-center exposure.

In [ ]:
dce_specs = [
    # Baseline 25 km DCE by period
    {
        "panel": "A. Baseline 25 km DCE by construction period",
        "variable": "ai_proximity_25km_before06",
        "description": "Original DCE, 25 km, data centers built before 2006",
    },
    {
        "panel": "A. Baseline 25 km DCE by construction period",
        "variable": "ai_proximity_25km_06_15",
        "description": "Original DCE, 25 km, data centers built in 2006-2015",
    },
    {
        "panel": "A. Baseline 25 km DCE by construction period",
        "variable": "ai_proximity_25km_16_19",
        "description": "Original DCE, 25 km, data centers built in 2016-2019",
    },
    {
        "panel": "A. Baseline 25 km DCE by construction period",
        "variable": "ai_proximity_25km_20_24",
        "description": "Original DCE, 25 km, data centers built in 2020-2024",
    },

    # Alternative spatial scales for the focal recent-period DCE
    {
        "panel": "B. Alternative spatial definitions for 2020-2024 DCE",
        "variable": "ai_proximity_15km_20_24",
        "description": "Original DCE, 15 km, 2020-2024",
    },
    {
        "panel": "B. Alternative spatial definitions for 2020-2024 DCE",
        "variable": "ai_proximity_50km_20_24",
        "description": "Original DCE, 50 km, 2020-2024",
    },
    {
        "panel": "B. Alternative spatial definitions for 2020-2024 DCE",
        "variable": "ai_proximity_100km_20_24",
        "description": "Original DCE, 100 km, 2020-2024",
    },
    {
        "panel": "B. Alternative spatial definitions for 2020-2024 DCE",
        "variable": "ai_proximity_200km_20_24",
        "description": "Original DCE, 200 km, 2020-2024",
    },
    {
        "panel": "B. Alternative spatial definitions for 2020-2024 DCE",
        "variable": "ai_gid1_20_24",
        "description": "Original DCE, same GID 1 region, 2020-2024",
    },
    {
        "panel": "B. Alternative spatial definitions for 2020-2024 DCE",
        "variable": "ai_gid2_20_24",
        "description": "Original DCE, same GID 2 region, 2020-2024",
    },
    {
        "panel": "B. Alternative spatial definitions for 2020-2024 DCE",
        "variable": "ai_ezone_20_24",
        "description": "Original DCE, same electricity zone, 2020-2024",
    },

    # Larger data centers
    {
        "panel": "C. Larger data-center exposure",
        "variable": "ai_larger_25km_20_24",
        "description": "Larger data-center DCE, 25 km, 2020-2024",
    },
]

check_vars(df, [x["variable"] for x in dce_specs], label="DCE variables")

dce_summary = summarize_numeric(df, dce_specs)

dce_summary_path = TABLES / "descriptive_dce_summary_statistics.csv"
dce_summary.to_csv(dce_summary_path, index=False)
print(f"Saved: {dce_summary_path}")

dce_summary

## 2. Nonzero DCE Density by Functional Form

This plot compares the pooled nonzero values of the focal 2020-2024 DCE under the baseline inverse-distance specification, minimum-distance caps, and winsorized variants.

In [ ]:
density_specs = [
    ("ai_proximity_25km_20_24", "Baseline 25 km"),
    ("ai_cap1km_25km_20_24", "Cap at 1 km"),
    ("ai_cap2km_25km_20_24", "Cap at 2 km"),
    ("ai_cap3km_25km_20_24", "Cap at 3 km"),
    ("ai_w95_25km_20_24", "Winsorized p95"),
    ("ai_w99_25km_20_24", "Winsorized p99"),
]

check_vars(df, [v for v, _ in density_specs], label="functional-form DCE variables")

functional_form_specs = [
    {
        "panel": "D. Functional-form variants for focal 2020-2024 DCE",
        "variable": v,
        "description": label,
    }
    for v, label in density_specs
]

functional_form_summary = summarize_numeric(df, functional_form_specs)
functional_form_summary_path = TABLES / "descriptive_dce_functional_form_nonzero_summary.csv"
functional_form_summary.to_csv(functional_form_summary_path, index=False)
print(f"Saved: {functional_form_summary_path}")

functional_form_summary

In [ ]:
plt.rcParams.update({
    "font.family": "sans-serif",
    "font.size": 10,
    "axes.linewidth": 0.6,
    "xtick.major.width": 0.6,
    "ytick.major.width": 0.6,
})

colors = {
    "Baseline 25 km": "#1F3A73",
    "Cap at 1 km": "#3E5F9F",
    "Cap at 2 km": "#6384C3",
    "Cap at 3 km": "#8A9FD0",
    "Winsorized p95": "#A786C8",
    "Winsorized p99": "#C1779E",
}

positive_values = {}
for var, label in density_specs:
    x = pd.to_numeric(df[var], errors="coerce")
    x = x[np.isfinite(x) & (x > 0)]
    positive_values[label] = x.to_numpy()

all_positive = np.concatenate([x for x in positive_values.values() if len(x) > 0])
if all_positive.size == 0:
    raise ValueError("No positive DCE values found for density plotting.")

# Use the pooled 99th percentile as the display window so the density is readable
# while keeping the full untrimmed summary statistics in the table above.
x_min = 0
x_max = np.nanpercentile(all_positive, 99)
if not np.isfinite(x_max) or x_max <= 0:
    x_max = np.nanmax(all_positive)

grid = np.linspace(x_min, x_max, 500)

fig, ax = plt.subplots()

for label, x in positive_values.items():
    x_plot = x[np.isfinite(x) & (x > 0)]
    if len(x_plot) < 3:
        continue

    if SCIPY_AVAILABLE:
        kde = gaussian_kde(x_plot)
        y = kde(grid)
        ax.plot(grid, y, color=colors[label], linewidth=1.2, label=label)
    else:
        ax.hist(
            x_plot,
            bins=60,
            range=(x_min, x_max),
            density=True,
            histtype="step",
            linewidth=1.2,
            color=colors[label],
            label=label,
        )

ax.set_xlim(x_min, x_max)
ax.set_xlabel("DCE value among nonzero observations")
ax.set_ylabel("Density")
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.grid(False)
ax.legend(frameon=False, fontsize=8, loc="upper right")

import matplotlib as mpl

mpl.rcParams["svg.fonttype"] = "none"
mpl.rcParams["pdf.fonttype"] = 42
mpl.rcParams["ps.fonttype"] = 42

fig.set_size_inches(6.4, 4.0)

fig_svg = FIGURES / "descriptive_dce_functional_form_nonzero_density.svg"
fig.savefig(fig_svg, bbox_inches="tight")

print(f"Saved: {fig_svg}")

plt.show()

## 3. Unit-Level Characteristics and Raw Country-Year Controls

Country-year controls are summarized using the raw non-imputed variables with the `_ni` suffix. If an `_ni` variable is unavailable, the notebook falls back to the base variable and reports that source variable in the output table.

In [ ]:
def raw_control_source(base_var):
    ni_var = f"{base_var}_ni"
    if ni_var in df.columns:
        return ni_var
    return base_var

unit_specs = [
    {
        "panel": "A. Unit-level coal plant characteristics",
        "variable": "Start_year",
        "description": "Commissioning year",
    },
    {
        "panel": "A. Unit-level coal plant characteristics",
        "variable": "time1",
        "description": "Plant age",
    },
    {
        "panel": "A. Unit-level coal plant characteristics",
        "variable": "Capacity_MW",
        "description": "Capacity (MW)",
    },
    {
        "panel": "A. Unit-level coal plant characteristics",
        "variable": "Emission_factor_kg_of_CO2_per_TJ",
        "description": "Emission factor (kg CO2/TJ)",
    },
    {
        "panel": "A. Unit-level coal plant characteristics",
        "variable": "Heat_rate_Btu_per_kWh",
        "description": "Heat rate (Btu/kWh)",
    },
]

country_year_base_specs = [
    ("rGDP_pc", "Real GDP per capita"),
    ("elec_price_db_uscent_kwh_raw", "Electricity price (US cents/kWh)"),
    ("td_loss_pct_output", "Transmission and distribution losses (% of output)"),
    ("demand_yoy_pct", "Electricity demand growth, year-on-year (%)"),
    ("renew_gen_share_pct", "Renewable generation share (%)"),
    ("policy_count_cum", "Cumulative climate-policy count"),
    ("nz_has_target_active", "Active net-zero target indicator"),
]

country_year_specs = []
for base_var, desc in country_year_base_specs:
    country_year_specs.append({
        "panel": "B. Raw country-year controls before imputation",
        "variable": base_var,
        "source_variable": raw_control_source(base_var),
        "description": desc,
    })

characteristic_specs = unit_specs + country_year_specs
check_vars(df, [x.get("source_variable", x["variable"]) for x in characteristic_specs], label="unit/control variables")

characteristics_summary = summarize_numeric(df, characteristic_specs)

characteristics_summary_path = TABLES / "descriptive_unit_and_raw_control_summary_statistics.csv"
characteristics_summary.to_csv(characteristics_summary_path, index=False)
print(f"Saved: {characteristics_summary_path}")

characteristics_summary

## Suggested Appendix Notes

**DCE table note.** The DCE variables are inverse-distance-weighted exposure measures computed at the coal-unit-year level. Nonzero shares report the fraction of non-missing observations with positive exposure. Functional-form variants for the 2020-2024 25 km DCE include minimum-distance caps and winsorized DCE values; summary statistics use the full distribution, while the density plot displays pooled nonzero observations.

**Characteristics table note.** Unit-level coal plant characteristics are reported from the final analysis data. Country-year controls are reported using their raw pre-imputation values, identified by the `_ni` suffix when available. In the regression analysis, missing country-year controls are filled using the interpolation and imputation procedure described in the Methods; robustness checks using alternative missing-data handling lead to the same substantive conclusions.